# NUMPY MASTERY — CHAPTER 22

```
NUMPY MASTERY
Current Chapter:  22 (Linear Algebra)
Progress:         22 / 75 chapters
Completed:        Ch1-21
Remaining:        53 chapters
```

This is where the math that actually powers ML and deep learning begins. Nearly every model —
linear regression, a neural network layer, an attention mechanism — is, at its core, one or more
matrix multiplications. Getting the shape rules and operator distinctions in this chapter
completely automatic is non-negotiable for anything past beginner ML work.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 22 — Linear Algebra

## Concept
NumPy gives you several related-but-different multiplication operations, and confusing them is
one of the most common sources of silently-wrong ML code:

| Operation | What it computes | Typical input shapes |
|---|---|---|
| `a * b` (element-wise) | Hadamard product — multiply corresponding elements | same shape (or broadcastable) |
| `np.dot(a, b)` | 1D: inner product (scalar). 2D: matrix multiplication. Higher-D: more exotic rules | varies |
| `a @ b` / `np.matmul(a, b)` | Matrix multiplication, with clear batch semantics | `(..., n, k) @ (..., k, m) -> (..., n, m)` |
| `np.inner(a, b)` | Sum of products over the last axis of each | last axes must match |
| `np.outer(a, b)` | All pairwise products — always returns 2D | any shapes, flattened first |
| `np.cross(a, b)` | 3D (or 2D) cross product (physics/geometry) | length-3 (or length-2) vectors |

## Why it matters
Every "forward pass" in a neural network is `output = input @ weights + bias`. Getting `@` vs
`*` backwards doesn't crash if shapes happen to broadcast — it silently computes something
completely different from what you intended.

## Industry use case
Computing predictions for a batch of samples: `X @ W + b` where `X` is `(n_samples,
n_features)` and `W` is `(n_features, n_outputs)` — this single line *is* the forward pass of a
linear model or a single dense neural network layer.


In [ ]:
# --- Element-wise (*) vs matrix multiplication (@) ---
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

elementwise = A * B          # Hadamard product: A[i,j] * B[i,j]
matmul = A @ B                # true matrix multiplication
dot_result = np.dot(A, B)     # same as A @ B for 2D arrays

print("A:\n", A)
print("B:\n", B)
print("\nA * B (element-wise)  :\n", elementwise)
print("\nA @ B (matrix mult)   :\n", matmul)
print("\nnp.dot(A, B)          :\n", dot_result)
print("\nA @ B == np.dot(A, B) for 2D?", np.array_equal(matmul, dot_result))


### Expected Output
```
A:
 [[1 2]
  [3 4]]
B:
 [[5 6]
  [7 8]]

A * B (element-wise)  :
 [[ 5 12]
  [21 32]]

A @ B (matrix mult)   :
 [[19 22]
  [43 50]]

np.dot(A, B)          :
 [[19 22]
  [43 50]]

A @ B == np.dot(A, B) for 2D? True
```

### Why the results differ, precisely
`A * B` computes `A[i,j] * B[i,j]` independently for every position — no summation, no notion
of rows/columns interacting. `A @ B` computes, for each output position `(i,j)`, the **dot
product of row i of A with column j of B**: `result[i,j] = sum_k A[i,k] * B[k,j]`. These are
fundamentally different operations that happen to require the same input shape here (`2x2`),
which is exactly why the `*` vs `@` bug is so easy to write and so hard to notice — no shape
error, just a wrong number.

### The shape rule for `@` / `matmul`
```
(n, k) @ (k, m) -> (n, m)     <- inner dimensions (k) must match; they "cancel out"
```


In [ ]:
# --- Shape requirements demo ---
X = np.arange(6).reshape(2, 3)    # 2 samples, 3 features -- (n_samples, n_features)
W = np.arange(12).reshape(3, 4)   # 3 features, 4 outputs  -- (n_features, n_outputs)

output = X @ W
print("X.shape:", X.shape, " W.shape:", W.shape, " (X @ W).shape:", output.shape)
print("X @ W:\n", output)

try:
    W @ X   # (3,4) @ (2,3) -- inner dims 4 and 2 don't match!
except ValueError as e:
    print("\nW @ X raised ValueError:", e)


### Expected Output
```
X.shape: (2, 3)  W.shape: (3, 4)  (X @ W).shape: (2, 4)
X @ W:
 [[20 23 26 29]
  [56 68 80 92]]

W @ X raised ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 4)
```

### This IS the neural network forward pass shape rule
`X` (2 samples × 3 features) `@` `W` (3 features × 4 output units) → `(2, 4)` — 2 samples, each
now represented by 4 output values. This exact pattern (`n_samples, n_features) @ (n_features,
n_units) -> (n_samples, n_units)`) is the forward computation of *every* dense/linear layer in
every neural network framework, and of linear regression's prediction step (Chapter 38).

### `@` vs `np.matmul` vs `np.dot` — when do they actually differ?
For plain 2D arrays, all three agree. They diverge for higher-dimensional ("batched") inputs:
- **`np.matmul`/`@`**: treats extra leading dimensions as a **batch** — `(batch, n, k) @
  (batch, k, m) -> (batch, n, m)`, multiplying each batch slice independently. This is what you
  want for batched neural network operations.
- **`np.dot`**: for arrays with more than 2 dimensions, follows a different, less intuitive
  rule (a sum product over the last axis of the first array and the second-to-last of the
  second) — it does **not** do batched matrix multiplication the way `matmul` does.

**Rule of thumb:** for 2D matrix multiplication, `@` (readable) and `np.matmul` are equivalent
and preferred; reserve `np.dot` for genuine 1D vector dot products; avoid `np.dot` for 3D+
"batched" data — use `@`/`matmul` there.


In [ ]:
# --- Batched matmul: the neural-network-relevant case ---
batch_X = np.random.default_rng(0).normal(size=(5, 2, 3))   # 5 batches, each a (2,3) matrix
batch_W = np.random.default_rng(1).normal(size=(5, 3, 4))   # 5 batches, each a (3,4) matrix

batch_result = batch_X @ batch_W    # multiplies EACH of the 5 (2,3)@(3,4) pairs independently
print("batch_X.shape:", batch_X.shape, " batch_W.shape:", batch_W.shape)
print("(batch_X @ batch_W).shape:", batch_result.shape, " <- one (2,4) result per batch item")

# Verify it really is independent per-batch multiplication, not something else:
manual_first = batch_X[0] @ batch_W[0]
print("Matches manual per-slice matmul for batch 0?", np.allclose(batch_result[0], manual_first))


### Expected Output
```
batch_X.shape: (5, 2, 3)  batch_W.shape: (5, 3, 4)
(batch_X @ batch_W).shape: (5, 2, 4)  <- one (2,4) result per batch item
Matches manual per-slice matmul for batch 0? True
```

### 1D vectors: `dot`, `inner`, `outer` — three genuinely different things


In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print("np.dot(u, v)   :", np.dot(u, v), "  <- scalar: sum(u[i]*v[i]) = 1*4+2*5+3*6")
print("u @ v          :", u @ v, "  <- same as dot for 1D vectors")
print("np.inner(u, v) :", np.inner(u, v), "  <- same as dot for 1D vectors")
print("np.outer(u, v) :\n", np.outer(u, v), "  <- ALL pairwise products, always 2D: shape (3,3)")

print("\nnp.cross(u, v) :", np.cross(u, v), "  <- 3D cross product (perpendicular to both u and v)")

M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("\nM:\n", M)
print("np.trace(M)    :", np.trace(M), "  <- sum of diagonal: 1+5+9")
print("np.diag(M)     :", np.diag(M), "  <- extract the diagonal as a vector")


### Expected Output
```
np.dot(u, v)   : 32   <- scalar: sum(u[i]*v[i]) = 1*4+2*5+3*6
u @ v          : 32   <- same as dot for 1D vectors
np.inner(u, v) : 32   <- same as dot for 1D vectors
np.outer(u, v) :
 [[ 4  5  6]
  [ 8 10 12]
  [12 15 18]]   <- ALL pairwise products, always 2D: shape (3,3)

np.cross(u, v) : [-3  6 -3]   <- 3D cross product (perpendicular to both u and v)

M:
 [[1 2 3]
  [4 5 6]
  [7 8 9]]
np.trace(M)    : 15   <- sum of diagonal: 1+5+9
np.diag(M)     : [1 5 9]   <- extract the diagonal as a vector
```

### Real-world use
- **`np.outer`**: building a rank-1 approximation, computing a co-occurrence-style matrix, or
  the outer-product broadcasting trick from Chapter 11.
- **`np.trace`**: appears in regularization terms (e.g., trace of a covariance matrix = total
  variance), and in some loss function formulations.
- **`np.diag`**: extracting variances from a covariance matrix, or constructing a diagonal
  matrix from a vector (dual use — same function name, input-dependent behavior, exactly like
  Chapter 22's `np.diag(vector)` vs `np.diag(matrix)` shown back in Chapter 2).

### Common Mistakes
- Using `*` when you meant `@` (or vice versa) — both run without error if shapes happen to be
  compatible for element-wise operation (e.g., equal shapes), silently producing the wrong
  numbers instead of crashing.
- Using `np.dot` on 3D+ batched arrays expecting `matmul`'s batching behavior — `dot` does not
  batch the way `matmul`/`@` do; always use `@`/`matmul` for batched neural-network-style data.
- Forgetting the **inner-dimension-must-match** rule and being surprised by a `ValueError` —
  always sanity check `(n, k) @ (k, m)` before writing the code, not after the error.

### Interview Insight
> "What's the difference between `A * B` and `A @ B` for two same-shaped square matrices?"
`A * B` is element-wise (Hadamard) multiplication — `result[i,j] = A[i,j] * B[i,j]`, no
summation. `A @ B` is true matrix multiplication — `result[i,j] = sum_k A[i,k] * B[k,j]`. They
can both run successfully on the same same-shaped square inputs, producing completely different
(and both "valid-looking") results — this is exactly why relying on "the code ran without an
error" is not sufficient validation for linear algebra code; always sanity-check against a hand
computation or a known small example.

> "Given a batch of images processed through 5 independent linear transformations, how would
> you compute all 5 results in one call?"
Stack the transformation matrices and inputs along a leading batch axis and use `@`/`np.matmul`
— it batches automatically over any leading dimensions, multiplying corresponding "slices"
independently, exactly like the `(5,2,3) @ (5,3,4) -> (5,2,4)` example above.


---
## Chapter 22 Summary

- `*` is element-wise multiplication (Hadamard product); `@`/`np.matmul`/`np.dot` (for 2D) are
  true matrix multiplication — confusing them doesn't crash on compatible shapes, it silently
  computes the wrong thing.
- Shape rule for matrix multiplication: `(n, k) @ (k, m) -> (n, m)` — inner dimensions must
  match and "cancel out." This is exactly the shape rule behind every neural network layer's
  forward pass: `(n_samples, n_features) @ (n_features, n_units) -> (n_samples, n_units)`.
- `@`/`np.matmul` batch automatically over leading dimensions (`(batch, n, k) @ (batch, k, m) ->
  (batch, n, m)`) — `np.dot` does **not** follow this same batching rule for 3D+ arrays; prefer
  `@`/`matmul` for any batched/ND linear algebra.
- `np.outer` always returns a full 2D pairwise-product matrix; `np.inner`/`np.dot` on 1D vectors
  both collapse to a scalar; `np.trace`/`np.diag` extract/construct diagonal information.

---

## Mastery Exercises

**Exercise 1 (Prediction):** Given `A` of shape `(4, 3)` and `B` of shape `(3, 5)`, predict the
shape of `A @ B`. Now predict whether `B @ A` is valid, and if so, its shape.

**Exercise 2:** For a batch of 10 samples with 8 features each (`X.shape == (10, 8)`) fed
through a linear layer with 3 output units, what shape must the weight matrix `W` be, and what
is the shape of `X @ W`? What shape must the bias vector `b` be for `X @ W + b` to broadcast
correctly (tie this back to Chapter 11)?

**Coding Challenge:** Implement `pairwise_dot_products(vectors)` where `vectors` has shape
`(n, d)` (n vectors of dimension d), returning the `(n, n)` matrix of all pairwise dot products
between vectors — without any Python-level loop. (Hint: this is a single matrix multiplication
— which one, and why?)

**Debugging Challenge:**
```python
weights = np.random.default_rng(0).normal(size=(4, 4))
gradients = np.random.default_rng(1).normal(size=(4, 4))
learning_rate = 0.01
weights_updated = weights - learning_rate * gradients   # gradient descent update
# a teammate "optimized" this to:
weights_updated_bug = weights @ (-learning_rate * gradients)   # what's wrong here?
```
Explain precisely why the second line is a completely different (and wrong) computation, even
though both run without error on these same-shaped `(4,4)` inputs.

**Interview Questions:**
1. Precisely define the difference between `*` and `@` for 2D arrays of the same shape.
2. State the shape compatibility rule for matrix multiplication.
3. How does `@`/`matmul` handle 3D+ "batched" arrays, and how does this differ from `np.dot`'s
   behavior on the same inputs?
4. What does `np.outer` compute, and how does its output shape differ from `np.inner`/`np.dot`
   on the same two 1D input vectors?

**Quiz:**
- Q1: True/False — `A * B` requires the same shape rule as `A @ B` (inner dimensions must
  match).
- Q2: For `u.shape == (5,)` and `v.shape == (5,)`, what is the shape of `np.outer(u, v)`?
- Q3: For `X.shape == (32, 10, 6)` and `W.shape == (32, 6, 2)`, what is the shape of `X @ W`?

---

